# Image verification lab ? Colab pixel-analysis baseline

This notebook runs in a **Google-hosted Python runtime**, avoiding the Windows runtime restriction on your laptop without changing its security settings. It is a notebook, not the local Streamlit website.

**Before running:** uploads go to Google's runtime. Only upload images you are comfortable processing there. Uploaded files and displayed images/results may persist in the runtime or saved notebook output; clear outputs before sharing. This notebook does not mount Google Drive or send images to Hugging Face. Hugging Face is contacted to download model files.

Use a new hosted CPU runtime (GPU is not needed). Run cells from top to bottom with each cell's play button. Wait for setup and model loading to finish. If rerunning setup after imports, restart the session and rerun from the top.

Two separate questions:
1. Does the image have camera-photo-like or AI-like pixel patterns?
2. Is there evidence of local editing? **Not implemented; inconclusive.**

The model cannot establish that an image or depicted event is legitimate. A camera photo can also be edited.


## 1. Install the CPU runtime dependencies
This changes packages only in the temporary Colab runtime. Downloads can take several minutes. The pinned CPU stack does not require torchvision; removing it avoids optional binary-extension conflicts.


In [ ]:
import subprocess
import sys

if 'google.colab' not in sys.modules:
    import importlib.util
    if importlib.util.find_spec('google.colab') is None:
        raise RuntimeError('Open this notebook in Google Colab using a hosted runtime.')

subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch==2.10.0+cpu', '--index-url', 'https://download.pytorch.org/whl/cpu'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers==4.57.6', 'Pillow>=11,<13'])
print('Setup complete. Continue to cell 2.')


## 2. Load the model
Baseline: [Organika/sdxl-detector](https://huggingface.co/Organika/sdxl-detector), pinned revision `b37fede`. Attribution: Organika / Colby / umm-maybe, as described in its model card. Listed license: CC-BY-NC-3.0, for educational/noncommercial use; review terms before other uses.

The model was fine-tuned on Wikimedia/SDXL pairs. Other generators, illustrations, screenshots, compression, and resizing can lead to wrong results. Its published metrics are not measured accuracy on your images. Weights are downloaded into this runtime; expect several hundred MB.


In [ ]:
"""Experimental pixel-only SDXL baseline; no EXIF fields enter the model."""

import argparse
import math
from pathlib import Path

from PIL import Image

MODEL_ID = "Organika/sdxl-detector"
MODEL_REVISION = "b37fede"
CACHE_DIR = Path.cwd() / ".models"


def load_detector(download=False):
    # Check this explicitly: torch._dynamo can swallow its DLL ImportError and
    # replace it with an unrelated-looking NP_SUPPORTED_MODULES import failure.
    import numpy.fft  # noqa: F401
    import torch
    from transformers import AutoImageProcessor, AutoModelForImageClassification

    options = dict(
        revision=MODEL_REVISION,
        cache_dir=str(CACHE_DIR),
        local_files_only=not download,
        trust_remote_code=False,
    )
    processor = AutoImageProcessor.from_pretrained(MODEL_ID, use_fast=False, **options)
    model = AutoModelForImageClassification.from_pretrained(
        MODEL_ID, use_safetensors=True, **options
    )
    labels = {int(key): value.lower() for key, value in model.config.id2label.items()}
    if set(labels.values()) != {"artificial", "human"}:
        raise ValueError("Unexpected model labels; inference stopped.")
    model.to("cpu").eval()
    return processor, model, torch, labels


def interpret_score(score):
    if not math.isfinite(score) or not 0 <= score <= 1:
        raise ValueError("Invalid model score.")
    # Exploratory bands only, not thresholds validated on the user's data.
    if score >= 0.9:
        return "AI-like signal (experimental)"
    if score <= 0.1:
        return "Camera-photo-like signal (experimental)"
    return "Inconclusive"


def analyze_pixels(image, detector):
    processor, model, torch, labels = detector
    # Make a new RGB image so EXIF and other metadata cannot enter inference.
    rgb = image.convert("RGB")
    pixels = Image.frombytes("RGB", rgb.size, rgb.tobytes())
    inputs = processor(images=pixels, return_tensors="pt")
    with torch.inference_mode():
        probabilities = model(**inputs).logits.softmax(dim=-1)[0]
    ai_index = next(index for index, name in labels.items() if name == "artificial")
    score = float(probabilities[ai_index].item())
    return {"ai_score": score, "signal": interpret_score(score)}



# This is a standalone copy of the project's detector functions.
# The actual model architecture comes from Transformers, not custom remote code.
detector = load_detector(download=True)
import torch
import transformers
import PIL
print('Model ready:', MODEL_ID, '@', MODEL_REVISION)
print('Python:', sys.version.split()[0])
print('torch:', torch.__version__, '| transformers:', transformers.__version__, '| Pillow:', PIL.__version__)


## 3. Upload and analyze JPEG/PNG images
Run this cell and select one or more files. Upload limits are checked after browser transfer, so choose files no larger than 20 MB. For another set of photos, rerun this cell.

A score near 1 favors the model's artificial class; near 0 favors its human class. This is **not a calibrated probability of authenticity**. The 0.10/0.90 bands are exploratory and have not been validated on your dataset. Overall authenticity stays inconclusive.


In [ ]:
import io
import warnings
from google.colab import files
from IPython.display import display
from PIL import Image, ImageOps, UnidentifiedImageError

results = []
uploaded = files.upload()
if not uploaded:
    print('No images selected. Run this cell again to upload.')

for filename, data in uploaded.items():
    print('\nFile:', filename)
    try:
        if len(data) > 20 * 1024 * 1024:
            raise ValueError('Choose an image no larger than 20 MB.')
        with warnings.catch_warnings():
            warnings.simplefilter('error', Image.DecompressionBombWarning)
            with Image.open(io.BytesIO(data)) as original:
                if original.format not in {'JPEG', 'PNG'}:
                    raise ValueError('Only JPEG and PNG contents are supported.')
                original.load()
                image_format, dimensions = original.format, original.size
                try:
                    image = ImageOps.exif_transpose(original).convert('RGB')
                except Exception:
                    image = original.convert('RGB')
        preview = image.copy()
        preview.thumbnail((1000, 1000))
        display(preview)
        print('Format:', image_format, '| Stored dimensions:', dimensions)
        result = analyze_pixels(image, detector)
        print('1. Generation model indication:', result['signal'])
        print('AI-class model score:', format(result['ai_score'], '.3f'))
        print('Authenticity conclusion: inconclusive ? baseline not validated on your target data.')
        print('2. Local editing: inconclusive ? edit localization is not implemented.')
        results.append({'filename': filename, **result,
                        'authenticity': 'inconclusive', 'local_editing': 'not implemented / inconclusive',
                        'model': MODEL_ID, 'revision': MODEL_REVISION})
    except (ValueError, OSError, SyntaxError, Image.DecompressionBombWarning) as error:
        print('Cannot analyze this image. Result: inconclusive.', str(error))
    except Exception as error:
        print('Analysis failed. Result: inconclusive.', type(error).__name__, str(error))

print('\nFinished. No detection score is a proof of authenticity.')


## 4. Optionally download a results report
This report records model scores and versions, not proven authenticity labels. Filenames are included. Only the latest upload batch is exported. Clear notebook outputs before sharing it if they contain private images or filenames.


In [ ]:
import json
from pathlib import Path

if not globals().get('results'):
    print('No successful results to export. Run the upload cell first.')
else:
    report = {
        'model': MODEL_ID, 'revision': MODEL_REVISION,
        'versions': {'python': sys.version.split()[0], 'torch': torch.__version__,
                     'transformers': transformers.__version__, 'pillow': PIL.__version__},
        'exploratory_bands': {'camera_photo_like_max': 0.10, 'ai_like_min': 0.90},
        'validated_on_target_data': False,
        'results': results,
    }
    Path('analysis_results.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    files.download('analysis_results.json')


## Next: scientific evaluation
A successful run checks functionality, not accuracy. Collect labeled camera photos and AI-generated images from multiple generators. Keep originals, edits, recompressed copies, and near duplicates in the same split to prevent leakage. Select thresholds on validation data, then freeze them before evaluating an untouched test set, including unseen generators.

Report precision, recall, F1, camera-photo false-positive rate, and results per source/generator. For inconclusive outcomes, report the inconclusive rate and coverage as well. Check calibration before calling scores probabilities. Keep local edit localization (TruFor) and signed history checks (C2PA) separate.

When finished, disconnect and delete the runtime using Colab's Runtime menu. Downloads and uploaded files may need to be supplied again in a new session.

References: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Colab file uploads](https://colab.research.google.com/notebooks/io.ipynb), [model card](https://huggingface.co/Organika/sdxl-detector), [TruFor](https://github.com/grip-unina/TruFor).


## Labeled diagnostic batch: rerun the six supplied examples
Run the setup and model-loading cells above first, then this cell. Upload all six examples. Labels below come from your stated image origins and are used **only after prediction for evaluation**; filenames never enter model inference. This batch diagnoses baseline failures; do not tune thresholds or claim general accuracy on these six images. Results can differ from the manually recorded rounded scores if image bytes or preprocessing differ. Upload originals, not screenshots of them. CSV export includes filenames.


In [ ]:
"""Evaluate labeled scores without loading a model or tuning on test examples."""

import argparse
import csv
import json
import math
from pathlib import Path


def evaluate(rows):
    records = list(rows)
    if not records:
        raise ValueError("No scored images supplied.")
    parsed = []
    for row in records:
        label = row["label"]
        score = float(row["ai_score"])
        if label not in {"camera", "generated"}:
            raise ValueError("Labels must be camera or generated.")
        if not math.isfinite(score) or not 0 <= score <= 1:
            raise ValueError("Scores must be finite and between 0 and 1.")
        parsed.append((label == "generated", score))
    positives = [score for label, score in parsed if label]
    negatives = [score for label, score in parsed if not label]
    tp = sum(label and score >= .5 for label, score in parsed)
    fp = sum(not label and score >= .5 for label, score in parsed)
    tn = sum(not label and score < .5 for label, score in parsed)
    fn = sum(label and score < .5 for label, score in parsed)
    decided = [(label, score) for label, score in parsed if score <= .1 or score >= .9]
    correct = sum(label == (score >= .9) for label, score in decided)
    ratio = lambda a, b: a / b if b else None
    auc = ratio(sum((p > n) + .5 * (p == n) for p in positives for n in negatives), len(positives) * len(negatives))
    return {
        "count": len(parsed), "generated_count": len(positives), "camera_count": len(negatives),
        "forced_cutoff": .5,
        "confusion_matrix": {"true_ai": tp, "false_ai": fp, "true_camera": tn, "missed_ai": fn},
        "accuracy": ratio(tp + tn, len(parsed)), "ai_precision": ratio(tp, tp + fp),
        "ai_recall": ratio(tp, tp + fn), "ai_f1": ratio(2 * tp, 2 * tp + fp + fn),
        "camera_false_positive_rate": ratio(fp, len(negatives)), "roc_auc": auc,
        "exploratory_bands": {"camera_max": .1, "ai_min": .9},
        "coverage": ratio(len(decided), len(parsed)),
        "inconclusive_count": len(parsed) - len(decided),
        "accuracy_among_decided": ratio(correct, len(decided)),
        "note": "Descriptive results only; no thresholds or weights fitted. Not a generalization estimate for a small convenience sample.",
    }



from google.colab import files
from PIL import Image, ImageOps
import io
import warnings

known_labels = {
    'CineMate AI Security Audit-2.png': 'generated', 'dog.jpg': 'generated',
    'school.jpg': 'generated', 'raw1.jpg': 'camera', 'raw2.jpg': 'camera', 'raw3.jpg': 'camera',
}
batch = files.upload()
scored = []
for filename, data in batch.items():
    if filename not in known_labels:
        print('Skipped: no supplied reference label for', filename)
        continue
    try:
        if len(data) > 20 * 1024 * 1024:
            raise ValueError('Image exceeds 20 MB')
        with warnings.catch_warnings():
            warnings.simplefilter('error', Image.DecompressionBombWarning)
            with Image.open(io.BytesIO(data)) as original:
                if original.format not in {'JPEG', 'PNG'}:
                    raise ValueError('JPEG/PNG required')
                if original.width * original.height > 20_000_000:
                    raise ValueError('Image exceeds 20 megapixels')
                original.load()
                try:
                    image = ImageOps.exif_transpose(original).convert('RGB')
                except Exception:
                    image = original.convert('RGB')
        prediction = analyze_pixels(image, detector)
        row = {'filename': filename, 'label': known_labels[filename], **prediction,
               'model': MODEL_ID, 'revision': MODEL_REVISION}
        scored.append(row)
        print(filename, '| reference:', row['label'], '| score:', round(row['ai_score'], 4), '|', row['signal'])
    except Exception as error:
        print('FAILED:', filename, type(error).__name__, str(error))

if scored:
    summary = evaluate(scored)
    print(json.dumps(summary, indent=2))
    missing = set(known_labels) - {row['filename'] for row in scored}
    if missing:
        print('INCOMPLETE batch. Missing/failed:', sorted(missing))
    with open('batch_scores.csv', 'w', newline='', encoding='utf-8') as output:
        writer = csv.DictWriter(output, fieldnames=list(scored[0]))
        writer.writeheader()
        writer.writerows(scored)
    files.download('batch_scores.csv')
else:
    print('No scored labeled examples. No accuracy report produced.')


## Compare a broader-generation candidate (experimental)
Your reference images were generated with Gemini and downloaded directly. The SDXL baseline's training coverage is a plausible mismatch, not a proven explanation for every failure.

Candidate: [s-gehring/detectra-openfake](https://huggingface.co/s-gehring/detectra-openfake), revision `c84565e`, CC-BY-NC-4.0, noncommercial use. Attribution: s-gehring, OpenFake authors, TensorFold, and Community Forensics authors (see model card). The author documents multi-generator training and Gemini evaluation. Published results are not validation on our images.

Run the six-image diagnostic batch cell above first. Then run the two cells below. They reuse those uploaded bytes and do not send images to Hugging Face. The candidate is a quantized ONNX model (~30 MB), run on CPU. It uses its own documented preprocessing; no filename or reference label enters inference.

We report raw sigmoid scores and the author's scaled score separately. The author's full multi-crop decision procedure is **not** implemented here; this is the single-center-crop baseline. Neither score is a calibrated authenticity probability for your dataset. We do not choose thresholds, average scores across models, or replace the deployed detector based on these six examples.


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'onnxruntime>=1.20,<2', 'huggingface_hub>=0.34,<1'])
"""Experimental ONNX candidate; compare before replacing the SDXL baseline.

Attribution and preprocessing: https://huggingface.co/s-gehring/detectra-openfake
Weights: CC-BY-NC-4.0. No target-data accuracy or calibration is established.
"""

import math

CANDIDATE_ID = "s-gehring/detectra-openfake"
CANDIDATE_REVISION = "c84565e"


def sigmoid(value):
    if not math.isfinite(value):
        raise ValueError("Non-finite model output")
    return 1 / (1 + math.exp(-value)) if value >= 0 else math.exp(value) / (1 + math.exp(value))


def prepare_pixels(image):
    import numpy as np
    from PIL import Image

    rgb = image.convert("RGB")
    width, height = rgb.size
    # Model card: shorter edge 440, bilinear; center crop 384; ImageNet norm.
    if width <= height:
        size = (440, int(440 * height / width))
    else:
        size = (int(440 * width / height), 440)
    resized = rgb.resize(size, Image.Resampling.BILINEAR)
    left, top = round((size[0] - 384) / 2), round((size[1] - 384) / 2)
    crop = resized.crop((left, top, left + 384, top + 384))
    pixels = np.asarray(crop, dtype=np.float32) / np.float32(255)
    mean = np.array([.485, .456, .406], dtype=np.float32)
    std = np.array([.229, .224, .225], dtype=np.float32)
    return np.ascontiguousarray(((pixels - mean) / std).transpose(2, 0, 1)[None])


def load_candidate():
    import onnxruntime as ort
    from huggingface_hub import hf_hub_download

    path = hf_hub_download(CANDIDATE_ID, "model.onnx", revision=CANDIDATE_REVISION)
    return ort.InferenceSession(path, providers=["CPUExecutionProvider"])


def candidate_score(image, session):
    import numpy as np

    inputs = session.get_inputs()
    if len(inputs) != 1 or inputs[0].name != "pixel_values":
        raise ValueError("Unexpected candidate input schema")
    output = np.asarray(session.run(["logits"], {"pixel_values": prepare_pixels(image)})[0])
    if output.size != 1:
        raise ValueError("Unexpected candidate output schema")
    logit = float(output.reshape(-1)[0])
    return {
        "ai_score": sigmoid(logit),
        "author_scaled_score": sigmoid(.552 * logit + .1933),
    }

candidate = load_candidate()
import onnxruntime
print('Candidate ready:', CANDIDATE_ID, CANDIDATE_REVISION, '| ONNX Runtime', onnxruntime.__version__)


In [ ]:
if not globals().get('batch') or not globals().get('scored'):
    raise RuntimeError('Run the labeled six-image diagnostic batch cell first.')

comparisons = []
for baseline in scored:
    filename = baseline['filename']
    try:
        with Image.open(io.BytesIO(batch[filename])) as original:
            original.load()
            try:
                image = ImageOps.exif_transpose(original).convert('RGB')
            except Exception:
                image = original.convert('RGB')
        candidate_result = candidate_score(image, candidate)
        row = {
            'filename': filename, 'label': baseline['label'],
            'sdxl_score': baseline['ai_score'],
            'candidate_raw_score': candidate_result['ai_score'],
            'candidate_author_scaled_score': candidate_result['author_scaled_score'],
            'candidate_model': CANDIDATE_ID, 'candidate_revision': CANDIDATE_REVISION,
        }
        comparisons.append(row)
        print(filename, '| reference:', row['label'], '| SDXL:', round(row['sdxl_score'], 4),
              '| candidate raw:', round(row['candidate_raw_score'], 4),
              '| author-scaled:', round(row['candidate_author_scaled_score'], 4))
    except Exception as error:
        print('CANDIDATE FAILED:', filename, type(error).__name__, str(error))

if comparisons:
    if len(comparisons) != len(known_labels):
        print('WARNING: incomplete paired batch; do not compare metrics from different image subsets.')
    for name, column in [('SDXL', 'sdxl_score'), ('Candidate raw', 'candidate_raw_score')]:
        report = evaluate([{'label': row['label'], 'ai_score': row[column]} for row in comparisons])
        print(name, json.dumps(report, indent=2))
    print('Metrics use the same paired images. Cutoffs are descriptive defaults, not tuned or validated operating points.')
    print('These known diagnostic cases cannot establish general accuracy. Keep an independent test set.')
    with open('model_comparison.csv', 'w', newline='', encoding='utf-8') as target:
        writer = csv.DictWriter(target, fieldnames=list(comparisons[0]))
        writer.writeheader()
        writer.writerows(comparisons)
    files.download('model_comparison.csv')
else:
    print('No paired results. No comparison claimed.')
